In [1]:
# =============================================================
# ROL 3 - Ensemble Model (Voting Classifier con modelos tuneados)
# =============================================================

import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


# =============================================================
# 1. Cargar dataset
# =============================================================
df = pd.read_csv("../data/processed/04_default_credit_features.csv")
df = df.drop(columns=["ID"])

TARGET = "default payment next month"
X = df.drop(columns=[TARGET])
y = df[TARGET]


# =============================================================
# 2. Split
# =============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# =============================================================
# 3. Preprocessing
# =============================================================
cat_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preproc = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


# =============================================================
# 4. Hiperparámetros
# =============================================================
svm_best_params = {
    "C": 0.515,
    "kernel": "rbf",
    "gamma": "scale"
}

gb_best_params = {
    "n_estimators": 69,
    "learning_rate": 0.087,
    "max_depth": 3
}

ada_best_params = {
    "n_estimators": 81,
    "learning_rate": 0.09
}


# =============================================================
# 5. Modelos base
# =============================================================
best_gb = GradientBoostingClassifier(**gb_best_params)
best_ada = AdaBoostClassifier(**ada_best_params)

# ✔ SOFT SVM (necesita probability=True)
best_svm_soft = SVC(**svm_best_params, probability=True)

# ✔ HARD SVM (NO duplicar probability)
best_svm_hard = SVC(**svm_best_params)


# =============================================================
# 6. Voting Soft
# =============================================================
voting_soft = VotingClassifier(
    estimators=[
        ("svm", best_svm_soft),
        ("gb", best_gb),
        ("ada", best_ada)
    ],
    voting="soft"
)

pipe_soft = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", voting_soft)
])


# =============================================================
# 7. Voting Hard
# =============================================================
voting_hard = VotingClassifier(
    estimators=[
        ("svm", best_svm_hard),
        ("gb", best_gb),
        ("ada", best_ada)
    ],
    voting="hard"
)

pipe_hard = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", voting_hard)
])


# =============================================================
# 8. Cross Validation
# =============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_soft = ["accuracy", "precision", "recall", "f1", "roc_auc"]
scoring_hard = ["accuracy", "precision", "recall", "f1"]  #  sin roc_auc



scores_soft = cross_validate(
    pipe_soft,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_soft,
    n_jobs=-1
)

scores_hard = cross_validate(
    pipe_hard,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_hard,
    n_jobs=-1
)


# =============================================================
# 9. Resultados
# =============================================================
print("=== SOFT VOTING ===")
for m in scores_soft:
    print(m, scores_soft[m].mean())

print("\n=== HARD VOTING ===")
for m in scores_hard:
    print(m, scores_hard[m].mean())

=== SOFT VOTING ===
fit_time 1059.9741993427276
score_time 36.35911831855774
test_accuracy 0.7792083333333334
test_precision 0.5009039062715726
test_recall 0.5954056774069874
test_f1 0.5440588528768536
test_roc_auc 0.7810565988005365

=== HARD VOTING ===
fit_time 189.05663084983826
score_time 14.717471933364868
test_accuracy 0.7787499999999999
test_precision 0.49995299961203266
test_recall 0.594462637848315
test_f1 0.5431114521486518


In [3]:
# =============================================================
# 10. BAGGING CLASSIFIER (Modelos con overfitting detectado)
# =============================================================

from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier

print("\n\n=================================================")
print("BAGGING CLASSIFIER - MODELOS CON OVERFITTING")
print("=================================================")

# =============================================================
# 1. Modelos base (OPTIMIZADOS PARA EVITAR MEMORY ERROR)
# =============================================================

rf_base = RandomForestClassifier(
    n_estimators=50,     
    random_state=42,
    n_jobs=-1
)

et_base = ExtraTreesClassifier(
    n_estimators=30,     
    max_depth=10,        
    random_state=42,
    n_jobs=1             
)

# =============================================================
# 2. Bagging models
# =============================================================

bagging_rf = BaggingClassifier(
    estimator=rf_base,
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)

bagging_et = BaggingClassifier(
    estimator=et_base,
    n_estimators=8,     
    random_state=42,
    n_jobs=1             
)

# =============================================================
# 3. Pipelines
# =============================================================

pipe_bag_rf = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", bagging_rf)
])

pipe_bag_et = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", bagging_et)
])

# =============================================================
# 4. Cross Validation
# =============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

scores_bag_rf = cross_validate(
    pipe_bag_rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

scores_bag_et = cross_validate(
    pipe_bag_et,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# =============================================================
# 5. Resultados
# =============================================================

print("\n=== BAGGING - RANDOM FOREST ===")
for m in scoring:
    print(m, scores_bag_rf[f"test_" + m].mean())

print("\n=== BAGGING - EXTRA TREES ===")
for m in scoring:
    print(m, scores_bag_et[f"test_" + m].mean())



BAGGING CLASSIFIER - MODELOS CON OVERFITTING

=== BAGGING - RANDOM FOREST ===
accuracy 0.7999583333333333
precision 0.5520419305390905
recall 0.5081955515796311
f1 0.529201743943324
roc_auc 0.7691427961424944

=== BAGGING - EXTRA TREES ===
accuracy 0.7774166666666666
precision 0.49744601628579277
recall 0.5916386665743685
f1 0.5404593081640137
roc_auc 0.7766536317362057


In [1]:
# =============================================================
# 11. GRADIENT BOOSTING AVANZADO (XGBoost + LightGBM)
# =============================================================
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
 
print("\n\n=================================================")
print("GRADIENT BOOSTING AVANZADO")
print("=================================================")
 
df = pd.read_csv("../data/processed/04_default_credit_features.csv")
df = df.drop(columns=["ID"])
 
TARGET = "default payment next month"
X = df.drop(columns=[TARGET])
y = df[TARGET]
 
# =============================================================
# 2. Split
# =============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
 
# =============================================================
# 3. Preprocessing
# =============================================================
cat_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
 
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
 
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
 
preproc = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])
 
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)
 
scoring_soft = ["accuracy", "precision", "recall", "f1", "roc_auc"]
 
 
# =============================================================
# 1. XGBOOST — búsqueda de hiperparámetros
# =============================================================
 
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False
)
 
pipe_xgb = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", xgb)
])
 
param_dist_xgb = {
    "model__n_estimators":    [100, 200, 300],
    "model__max_depth":       [3, 5, 6, 8],
    "model__learning_rate":   [0.01, 0.05, 0.1, 0.2],
    "model__subsample":       [0.6, 0.8, 1.0],
    "model__colsample_bytree":[0.6, 0.8, 1.0]
}
 
search_xgb = RandomizedSearchCV(
    pipe_xgb,
    param_distributions=param_dist_xgb,
    n_iter=20,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)
search_xgb.fit(X_train, y_train)
 
print("\n=== XGBOOST — Mejores hiperparámetros ===")
print(search_xgb.best_params_)
 
# Pipeline final con los mejores parámetros encontrados
best_xgb_params = {k.replace("model__", ""): v for k, v in search_xgb.best_params_.items()}
 
xgb_best = XGBClassifier(
    **best_xgb_params,
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False
)
 
pipe_xgb = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", xgb_best)
])
 
scores_xgb = cross_validate(
    pipe_xgb,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_soft,
    n_jobs=-1
)
 
 
# =============================================================
# 2. LIGHTGBM — búsqueda de hiperparámetros
# =============================================================
 
lgbm = LGBMClassifier(
    random_state=42,
    verbose=-1
)
 
pipe_lgbm = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", lgbm)
])
 
param_dist_lgbm = {
    "model__n_estimators":    [100, 200, 300],
    "model__max_depth":       [3, 5, 6, 8],
    "model__learning_rate":   [0.01, 0.05, 0.1, 0.2],
    "model__subsample":       [0.6, 0.8, 1.0],
    "model__colsample_bytree":[0.6, 0.8, 1.0],
    "model__num_leaves":      [31, 50, 70]
}
 
search_lgbm = RandomizedSearchCV(
    pipe_lgbm,
    param_distributions=param_dist_lgbm,
    n_iter=20,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)
search_lgbm.fit(X_train, y_train)
 
print("\n=== LIGHTGBM — Mejores hiperparámetros ===")
print(search_lgbm.best_params_)
 
# Pipeline final con los mejores parámetros encontrados
best_lgbm_params = {k.replace("model__", ""): v for k, v in search_lgbm.best_params_.items()}
 
lgbm_best = LGBMClassifier(
    **best_lgbm_params,
    random_state=42,
    verbose=-1
)
 
pipe_lgbm = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", lgbm_best)
])
 
scores_lgbm = cross_validate(
    pipe_lgbm,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_soft,
    n_jobs=-1
)
# =============================================================
# 3. RESULTADOS
# =============================================================

print("\n=== XGBOOST RESULTS ===")
print("Accuracy:",  scores_xgb["test_accuracy"].mean())
print("Precision:", scores_xgb["test_precision"].mean())
print("Recall:",    scores_xgb["test_recall"].mean())
print("F1:",        scores_xgb["test_f1"].mean())
print("ROC-AUC:",   scores_xgb["test_roc_auc"].mean())

print("\n=== LIGHTGBM RESULTS ===")
print("Accuracy:",  scores_lgbm["test_accuracy"].mean())
print("Precision:", scores_lgbm["test_precision"].mean())
print("Recall:",    scores_lgbm["test_recall"].mean())
print("F1:",        scores_lgbm["test_f1"].mean())
print("ROC-AUC:",   scores_lgbm["test_roc_auc"].mean())



GRADIENT BOOSTING AVANZADO


C:\Users\jesus\miniforge3\envs\proyecto_python\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:02:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== XGBOOST — Mejores hiperparámetros ===
{'model__subsample': 0.6, 'model__n_estimators': 200, 'model__max_depth': 8, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.6}

=== LIGHTGBM — Mejores hiperparámetros ===
{'model__subsample': 1.0, 'model__num_leaves': 70, 'model__n_estimators': 200, 'model__max_depth': 5, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.6}

=== XGBOOST RESULTS ===
Accuracy: 0.7795
Precision: 0.5014881987830848
Recall: 0.5944637028280536
F1: 0.5439898575388498
ROC-AUC: 0.7811657284665157

=== LIGHTGBM RESULTS ===
Accuracy: 0.7731250000000001
Precision: 0.4898445342140138
Recall: 0.6114165827995123
F1: 0.5438789356045541
ROC-AUC: 0.7806951083478613


In [15]:
# =============================================================
# 12. STACKING CLASSIFIER
# =============================================================

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
# =============================================================
# IMPORTS NECESARIOS PARA STACKING
# =============================================================

from sklearn.svm import SVC
from sklearn.ensemble import (
    GradientBoostingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

print("\n\n=================================================")
print("STACKING CLASSIFIER")
print("=================================================")

# =============================================================
# 1. Modelos base
# =============================================================

# SVM
stack_svm = SVC(
    C=0.515,
    kernel="rbf",
    gamma="scale",
    probability=True,
    random_state=42
)

# Gradient Boosting
stack_gb = GradientBoostingClassifier(
    n_estimators=69,
    learning_rate=0.087,
    max_depth=3,
    random_state=42
)

# XGBoost
stack_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

# LightGBM
stack_lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

# =============================================================
# 2. Meta-learner
# =============================================================

meta_learner = LogisticRegression(
    random_state=42,
    max_iter=1000
)

# =============================================================
# 3. Stacking Classifier
# =============================================================

stacking_model = StackingClassifier(
    estimators=[
        ("svm", stack_svm),
        ("gb", stack_gb),
        ("xgb", stack_xgb),
        ("lgbm", stack_lgbm)
    ],
    final_estimator=meta_learner,
    cv=cv,
    n_jobs=-1
)

# =============================================================
# 4. Pipeline
# =============================================================

pipe_stacking = ImbPipeline([
    ("preprocessing", preproc),
    ("sampling", RandomOverSampler(random_state=42)),
    ("model", stacking_model)
])

# =============================================================
# 5. Cross Validation
# =============================================================

scores_stacking = cross_validate(
    pipe_stacking,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_soft,
    n_jobs=-1
)

# =============================================================
# 6. Resultados
# =============================================================

print("\n=== STACKING RESULTS ===")

print("Accuracy:", scores_stacking["test_accuracy"].mean())
print("Precision:", scores_stacking["test_precision"].mean())
print("Recall:", scores_stacking["test_recall"].mean())
print("F1:", scores_stacking["test_f1"].mean())
print("ROC-AUC:", scores_stacking["test_roc_auc"].mean())



STACKING CLASSIFIER

=== STACKING RESULTS ===
Accuracy: 0.7729583333333333
Precision: 0.4869953595889836
Recall: 0.493317252139278
F1: 0.4901240444920953
ROC-AUC: 0.7283136992178432


#**CUADRO COMPARATIVO FINAL ( BASELINES VS ENSAMBLES)**

| Ranking | Modelo | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---------|--------|----------|-----------|--------|----------|---------|
| 🥇 | Soft Voting | 0.7792 | 0.5009 | 0.5954 | 0.5440 | 0.7810 |
| 🥇 | XGBoost (tuned) | 0.7795 | 0.5015 | 0.5945 | 0.5440 | 0.7812 |
| 🥈 | LightGBM (tuned) | 0.7731 | 0.4898 | 0.6114 | 0.5439 | 0.7807 |
| 🥉 | Extra Trees (Bagging) | 0.7774 | 0.4974 | 0.5916 | 0.5404 | 0.7766 |
| 4 | Hard Voting | 0.7787 | 0.4999 | 0.5944 | 0.5431 | — |
| 5 | Extra Trees (baseline tuned) | 0.7770 | 0.4970 | 0.5920 | 0.5400 | 0.7770 |
| 6 | LightGBM (sin tunear) | 0.7723 | 0.4878 | 0.5797 | 0.5298 | 0.7707 |
| 7 | Random Forest (Bagging) | 0.7999 | 0.5520 | 0.5082 | 0.5292 | 0.7691 |
| 8 | Random Forest (baseline tuned) | 0.8000 | 0.5520 | 0.5080 | 0.5290 | 0.7690 |
| 9 | XGBoost (sin tunear) | 0.7760 | 0.4946 | 0.5635 | 0.5268 | 0.7664 |
| 10 | Gradient Boosting (tuned) | 0.7760 | 0.4980 | 0.5650 | 0.5270 | 0.7650 |
| 11 | SVM (tuned) | 0.7750 | 0.4950 | 0.5600 | 0.5250 | 0.7600 |
| 12 | Stacking | 0.7730 | 0.4870 | 0.4933 | 0.4901 | 0.7283 |

**DOCUMENTACION DE COMBINACIONES QUE FUNCIONAN**

| Tipo de combinación         | Modelos incluidos                                               | Observaciones principales                                                                                                 |
| --------------------------- | --------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------- |
| **Soft Voting**             | SVM, Gradient Boosting, AdaBoost                                | Mejor balance entre precision, recall y F1; combina modelos con diferentes fortalezas; logra el **mejor ROC-AUC (0.781)** |
| **Hard Voting**             | SVM, Gradient Boosting, AdaBoost                                | Similar al Soft Voting, pero slightly peor F1; no aprovecha probabilidades de los modelos                                 |
| **Bagging - Random Forest** | RF base + Bagging                                               | Alta accuracy (0.800) y precision (0.552); robusto a overfitting; recall moderado (0.508)                                 |
| **Bagging - Extra Trees**   | ET base + Bagging                                               | Muy buen recall (0.592) y F1 (0.540); estable; ROC-AUC alto (0.777)                                                       |
| **Boosting - LightGBM**     | LightGBM                                                        | Rendimiento consistente en ROC-AUC y F1; no supera ensambles                                                              |
| **Boosting - XGBoost**      | XGBoost                                                         | Similar a LightGBM; ligeramente menor F1 y ROC-AUC                                                                        |
| **Stacking**                | SVM, Gradient Boosting, XGBoost, LightGBM + Logistic Regression | No mejora respecto a baselines; baja diversidad entre modelos base y meta-learner simple; ROC-AUC más bajo (0.728)        |
